# Eksperimen 2b: Validasi Ground Truth Labeling Berbasis Jurnal Biomekanika Terbaru (2022-2024)

Notebook ini memvalidasi secara matematis apakah setiap video yang diberi label **"Benar"** (`label=0`)
benar-benar memenuhi kriteria biomekanik kuantitatif dari literatur ilmiah terbaru.

**Referensi yang digunakan:**
1. **Chen, K.-Y. et al. (2022)** — *"Fitness Movement Types and Completeness Detection Using a Transfer-Learning-Based Deep Neural Network."*
2. **Rao, P., Asha, C. S., & Rao, R. P. (2023)** — *"Real-time Posture Correction of Squat Exercise: A Deep Learning Approach for Performance Analysis and Error Correction."*
3. **Ko, Y.-M., Nasridinov, A., & Park, S.-H. (2024)** — *"Real-Time AI Posture Correction for Powerlifting Exercises Using YOLOv5 and MediaPipe."*

**Validator yang dijalankan per gerakan:**

| Gerakan | Kriteria yang Diuji | Threshold | Referensi |
|---|---|---|---|
| **Squat** | Sudut Lutut (Depth) | ≤ 100° | Chen et al. (2022) |
| **Squat** | Knee Valgus (lebar lutut/kaki) | Rasio ≥ 0.85 | Rao et al. (2023) |
| **Bench Press** | Sudut Siku (Elbow ROM) | ≤ 85° | Ko et al. (2024) |
| **Deadlift** | Inklinasi Punggung dari Vertikal | 20° – 60° | Chen et al. (2022), Ko et al. (2024) |

> **Catatan:** Video berlabel "Benar" yang **gagal** divalidasi akan ditandai sebagai peringatan dan perlu dikaji ulang (re-label atau re-shot) sebelum digunakan untuk pelatihan model.

## 1. Import Library & Konfigurasi

In [1]:
# ============================================================
# Import library dan daftarkan src/ ke sys.path
# ============================================================
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Tambahkan direktori src/ ke Python path
sys.path.insert(0, os.path.abspath("../src"))

from data.biomechanics_validator import BiomechanicalValidator

print("Import berhasil.")

# Inisialisasi validator
validator = BiomechanicalValidator()

print("BiomechanicalValidator berhasil diinisialisasi.")
print(f"  Threshold Squat Depth    : ≤ {validator.SQUAT_DEPTH_THRESHOLD_DEG}°")
print(f"  Threshold Knee Valgus    : Rasio ≥ {validator.SQUAT_VALGUS_RATIO_THRESHOLD}")
print(f"  Threshold Bench Elbow    : ≤ {validator.BENCH_ELBOW_THRESHOLD_DEG}°")
print(f"  Threshold Deadlift Spine : {validator.DEADLIFT_SPINE_MIN_DEG}° – {validator.DEADLIFT_SPINE_MAX_DEG}°")

Import berhasil.
BiomechanicalValidator berhasil diinisialisasi.
  Threshold Squat Depth    : ≤ 100.0°
  Threshold Knee Valgus    : Rasio ≥ 0.85
  Threshold Bench Elbow    : ≤ 85.0°
  Threshold Deadlift Spine : 20.0° – 60.0°


## 2. Definisi Path & Peta Gerakan

In [2]:
# ============================================================
# Definisi path dan peta nama gerakan ke fungsi validator
# ============================================================
MANIFEST_PATH = Path("../data/processed/dataset_manifest.csv")

# Peta nama folder gerakan (case-insensitive) ke metode validator yang sesuai
# Kunci: substring nama file .npy (format dari build_dataset.py: ExerciseName_Kelas_NNN.npy)
EXERCISE_VALIDATOR_MAP = {
    "squat"      : validator.validate_squat,
    "deadlift"   : validator.validate_deadlift,
    "benchpress" : validator.validate_benchpress,
    "bench"      : validator.validate_benchpress,  # Alias untuk variasi penamaan folder
}

print(f"Manifest path  : {MANIFEST_PATH.resolve()}")
print(f"Gerakan yang didukung validator: {list(EXERCISE_VALIDATOR_MAP.keys())}")

# Cek ketersediaan manifest
if not MANIFEST_PATH.exists():
    display(HTML(
        '<span style="color:orange; font-weight:bold;">'
        '[PERINGATAN] Manifest CSV tidak ditemukan!<br>'
        'Jalankan build_dataset.py atau notebook 04 terlebih dahulu.'
        '</span>'
    ))
else:
    df_manifest = pd.read_csv(MANIFEST_PATH)
    n_benar = (df_manifest["label"] == 0).sum()
    n_salah = (df_manifest["label"] == 1).sum()
    print(f"\n[OK] Manifest ditemukan: {len(df_manifest)} sampel total")
    print(f"     Label 0 (Benar) : {n_benar} sampel  ← yang akan divalidasi")
    print(f"     Label 1 (Salah) : {n_salah} sampel  ← dilewati")

Manifest path  : D:\Data-Aji\KULIAH\Tugas-Akhir\AttentiveSkel3D-WeightTraining-PoC\data\processed\dataset_manifest.csv
Gerakan yang didukung validator: ['squat', 'deadlift', 'benchpress', 'bench']

[OK] Manifest ditemukan: 4 sampel total
     Label 0 (Benar) : 2 sampel  ← yang akan divalidasi
     Label 1 (Salah) : 2 sampel  ← dilewati


## 3. Jalankan Validasi Biomekanik pada Semua Sampel "Benar"

In [3]:
# ============================================================
# Loop semua baris di manifest dengan label=0 (Benar).
# Untuk setiap sampel:
#   1. Deteksi jenis gerakan dari nama file
#   2. Muat tensor .npy
#   3. Jalankan validator yang sesuai
#   4. Tampilkan peringatan MERAH jika validasi gagal
# ============================================================

if not MANIFEST_PATH.exists():
    print("[STOP] Manifest tidak tersedia. Jalankan cell sebelumnya terlebih dahulu.")
else:
    df_manifest = pd.read_csv(MANIFEST_PATH)

    # Filter hanya sampel berlabel "Benar" (label=0)
    df_benar = df_manifest[df_manifest["label"] == 0].reset_index(drop=True)

    print(f"Memvalidasi {len(df_benar)} sampel berlabel 'Benar'...\n")
    print("=" * 72)

    # List untuk merekam hasil tiap sampel
    results = []

    for row_idx, row in df_benar.iterrows():
        file_path = row["file_path"]
        filename  = Path(file_path).stem.lower()  # Contoh: "deadlift_benar_001"

        # ── Deteksi jenis gerakan dari nama file ─────────────────────────────
        detected_exercise = None
        validate_fn       = None

        for exercise_key, fn in EXERCISE_VALIDATOR_MAP.items():
            if exercise_key in filename:
                detected_exercise = exercise_key
                validate_fn       = fn
                break

        # Jika jenis gerakan tidak dikenali, lewati
        if validate_fn is None:
            display(HTML(
                f'<span style="color:orange;">'
                f'[SKIP] {Path(file_path).name}: Jenis gerakan tidak dikenali dari nama file. '
                f'Pastikan nama folder latihan mengandung salah satu dari: '
                f'{list(EXERCISE_VALIDATOR_MAP.keys())}'
                f'</span>'
            ))
            results.append({
                "file": Path(file_path).name,
                "exercise": "tidak dikenali",
                "is_valid": None,
                "reason": "Jenis gerakan tidak dikenali"
            })
            continue

        # ── Muat tensor .npy dari disk ────────────────────────────────────────
        if not os.path.exists(file_path):
            display(HTML(
                f'<span style="color:orange;">'
                f'[SKIP] {Path(file_path).name}: File .npy tidak ditemukan di disk.'
                f'</span>'
            ))
            results.append({
                "file": Path(file_path).name,
                "exercise": detected_exercise,
                "is_valid": None,
                "reason": "File .npy tidak ditemukan"
            })
            continue

        tensor_data = np.load(file_path)  # Diharapkan (64, 33, 3)

        # ── Jalankan validator biomekanik ────────────────────────────────────
        try:
            is_valid, reason = validate_fn(tensor_data)
        except Exception as e:
            is_valid = False
            reason   = f"Error saat menjalankan validator: {e}"

        results.append({
            "file"      : Path(file_path).name,
            "exercise"  : detected_exercise,
            "is_valid"  : is_valid,
            "reason"    : reason,
        })

        # ── Tampilkan hasil ───────────────────────────────────────────────────
        if is_valid:
            display(HTML(
                f'<span style="color:green;">'
                f'[VALID] {Path(file_path).name} ({detected_exercise.upper()}) '
                f'→ {reason}'
                f'</span>'
            ))
        else:
            # Peringatan merah: video berlabel Benar tapi gagal validasi matematis
            display(HTML(
                f'<span style="color:red; font-weight:bold;">'
                f'[WARNING] Video {file_path} berlabel BENAR tetapi secara matematis '
                f'gagal divalidasi. Alasan: {reason}'
                f'</span>'
            ))

    print("=" * 72)
    print(f"Validasi selesai.")

Memvalidasi 2 sampel berlabel 'Benar'...



Validasi selesai.


## 4. Ringkasan Hasil Validasi

In [4]:
# ============================================================
# Tampilkan tabel ringkasan dan statistik akhir validasi
# ============================================================
if "results" not in dir() or not results:
    print("Tidak ada hasil validasi. Jalankan cell di atas terlebih dahulu.")
else:
    df_results = pd.DataFrame(results)

    # Hitung statistik ringkasan
    total      = len(df_results)
    valid      = df_results["is_valid"].sum()
    invalid    = (df_results["is_valid"] == False).sum()
    skipped    = df_results["is_valid"].isna().sum()

    print("RINGKASAN HASIL VALIDASI BIOMEKANIK")
    print("=" * 50)
    print(f"Total sampel 'Benar' yang diperiksa : {total}")
    print(f"  ✓ VALID                            : {valid}")
    print(f"  ✗ GAGAL (perlu dikaji ulang)        : {invalid}")
    print(f"  ○ DILEWATI (tidak diproses)         : {skipped}")
    print()

    if invalid > 0:
        display(HTML(
            f'<b style="color:red;">⚠ Terdapat {invalid} sampel yang perlu dikaji ulang!</b><br>'
            f'Video tersebut diberi label "Benar" namun secara matematis tidak memenuhi '
            f'kriteria biomekanik dari literatur. Pertimbangkan untuk:<br>'
            f'&nbsp;&nbsp;1. Re-shoot video dengan form yang benar<br>'
            f'&nbsp;&nbsp;2. Re-label menjadi "Salah" jika form memang tidak benar<br>'
            f'&nbsp;&nbsp;3. Adjust threshold jika ada perbedaan interpretasi dengan literatur'
        ))
    else:
        display(HTML(
            '<b style="color:green;">✓ Semua sampel berlabel "Benar" lulus validasi biomekanik!</b>'
        ))

    print()
    print("Detail per sampel:")
    print(df_results.to_string(index=False))

RINGKASAN HASIL VALIDASI BIOMEKANIK
Total sampel 'Benar' yang diperiksa : 2
  ✓ VALID                            : 2
  ✗ GAGAL (perlu dikaji ulang)        : 0
  ○ DILEWATI (tidak diproses)         : 0




Detail per sampel:
                  file exercise  is_valid                                                                                                                         reason
Deadlift_Benar_002.npy deadlift      True Deadlift valid. Inklinasi punggung maksimum = 20.1° (20.0° ≤ x ≤ 60.0°), hip hinge pattern yang aman terdeteksi pada frame 37.
Deadlift_Benar_001.npy deadlift      True Deadlift valid. Inklinasi punggung maksimum = 58.0° (20.0° ≤ x ≤ 60.0°), hip hinge pattern yang aman terdeteksi pada frame 13.


## 5. Verifikasi Metode `calculate_angle_3d`

In [5]:
# ============================================================
# Unit test dasar untuk memverifikasi perhitungan sudut 3D
# menggunakan titik-titik dengan sudut yang diketahui.
# ============================================================
print("Verifikasi metode calculate_angle_3d:")
print("-" * 45)

test_cases = [
    # (a, b, c, sudut_diharapkan, deskripsi)
    (np.array([1, 0, 0]), np.array([0, 0, 0]), np.array([0, 1, 0]),  90.0,  "Sudut siku-siku (90°)"),
    (np.array([1, 0, 0]), np.array([0, 0, 0]), np.array([1, 0, 0]),   0.0,  "Sudut nol (0°) — titik a=c"),
    (np.array([1, 0, 0]), np.array([0, 0, 0]), np.array([-1,0, 0]), 180.0,  "Sudut lurus (180°)"),
    (np.array([1, 1, 0]), np.array([0, 0, 0]), np.array([1, 0, 0]),  45.0,  "Sudut 45°"),
]

all_passed = True
for a, b, c, expected, desc in test_cases:
    computed = validator.calculate_angle_3d(a, b, c)
    passed   = abs(computed - expected) < 0.01
    status   = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"  [{status}] {desc}: dihitung={computed:.2f}°, diharapkan={expected:.2f}°")

print()
if all_passed:
    display(HTML('<b style="color:green;">[OK] Semua test perhitungan sudut lulus.</b>'))
else:
    display(HTML('<b style="color:red;">[FAIL] Ada test yang gagal. Periksa implementasi calculate_angle_3d.</b>'))

Verifikasi metode calculate_angle_3d:
---------------------------------------------
  [PASS] Sudut siku-siku (90°): dihitung=90.00°, diharapkan=90.00°
  [PASS] Sudut nol (0°) — titik a=c: dihitung=0.00°, diharapkan=0.00°
  [PASS] Sudut lurus (180°): dihitung=180.00°, diharapkan=180.00°
  [PASS] Sudut 45°: dihitung=45.00°, diharapkan=45.00°



## 6. Visualisasi: Profil Sudut Lutut (Squat) Sepanjang 64 Frame

In [6]:
# ============================================================
# Untuk setiap sampel Squat berlabel Benar, plot profil sudut
# lutut (kiri, kanan, rata-rata) sepanjang 64 frame agar bisa
# diamati pola gerakannya secara visual.
# ============================================================
if not MANIFEST_PATH.exists():
    print("Manifest tidak tersedia.")
else:
    df_manifest = pd.read_csv(MANIFEST_PATH)
    squat_benar = df_manifest[
        (df_manifest["label"] == 0) &
        (df_manifest["file_path"].str.lower().str.contains("squat"))
    ].reset_index(drop=True)

    if len(squat_benar) == 0:
        print("Tidak ada sampel Squat berlabel Benar di manifest.")
        print("Pastikan nama folder latihan mengandung kata 'Squat'.")
    else:
        n_plot = len(squat_benar)
        fig, axes = plt.subplots(1, n_plot, figsize=(7 * n_plot, 5), squeeze=False)
        fig.suptitle("Profil Sudut Lutut — Sampel Squat Berlabel 'Benar'", fontsize=13)

        for i, (_, row) in enumerate(squat_benar.iterrows()):
            if not os.path.exists(row["file_path"]):
                continue

            tensor = np.load(row["file_path"])  # (64, 33, 3)

            # Hitung sudut Hip-Knee-Ankle per frame (kiri dan kanan)
            angles_left  = validator._get_per_frame_angles(
                tensor,
                BiomechanicalValidator.IDX_L_HIP,
                BiomechanicalValidator.IDX_L_KNEE,
                BiomechanicalValidator.IDX_L_ANKLE,
            )
            angles_right = validator._get_per_frame_angles(
                tensor,
                BiomechanicalValidator.IDX_R_HIP,
                BiomechanicalValidator.IDX_R_KNEE,
                BiomechanicalValidator.IDX_R_ANKLE,
            )
            angles_avg = (angles_left + angles_right) / 2.0
            deepest_frame = int(np.argmin(angles_avg))

            ax = axes[0][i]
            frames = range(tensor.shape[0])
            ax.plot(frames, angles_left,  "b--", alpha=0.6, label="Lutut Kiri")
            ax.plot(frames, angles_right, "r--", alpha=0.6, label="Lutut Kanan")
            ax.plot(frames, angles_avg,   "k-",  linewidth=2, label="Rata-rata")

            # Tandai posisi terdalam dan threshold
            ax.axvline(x=deepest_frame, color="green", linestyle=":", linewidth=1.5,
                       label=f"Posisi terdalam (frame {deepest_frame})")
            ax.axhline(y=BiomechanicalValidator.SQUAT_DEPTH_THRESHOLD_DEG,
                       color="orange", linestyle="--", linewidth=1.2,
                       label=f"Threshold ({BiomechanicalValidator.SQUAT_DEPTH_THRESHOLD_DEG}°)")

            ax.set_title(Path(row["file_path"]).name, fontsize=9)
            ax.set_xlabel("Frame (0–63)")
            ax.set_ylabel("Sudut Lutut (derajat)")
            ax.set_ylim(0, 200)
            ax.legend(fontsize=7)
            ax.grid(True, linestyle="--", alpha=0.4)

        plt.tight_layout()
        plt.show()

Tidak ada sampel Squat berlabel Benar di manifest.
Pastikan nama folder latihan mengandung kata 'Squat'.
